# Phase 2: Reasoning Tasks Feasibility Check

Before building the distillation loop, we must verify that our chosen model (`SmolLM2-1.7B-Instruct`) actually exhibits the required behavior on our reasoning tasks:
1. **Teacher Competence**: The model must be able to solve the task when given 4 demonstrations.
2. **Student Ignorance**: The model must fail to solve the task when given 0 demonstrations (0-shot).

If the teacher fails, there is no signal to distill. If the 0-shot student succeeds, there is no gap to close.
We will evaluate all 20 tasks from `forsmol.json` and `forsmol2.json` to find the tasks with the strongest distillation potential.

In [8]:
!pip install transformers torch

## Step 1 — Setup and Load Model

In [9]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(model_name)

# We only need one model for this feasibility check
if 'model' not in globals():
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
    ).to(device)
model.eval()

print(f"Model {model_name} loaded in bfloat16.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model HuggingFaceTB/SmolLM2-1.7B-Instruct loaded in bfloat16.


## Step 2 — Load Datasets

In [39]:
import json
raw_tasks_json = r"""[
  {
    "id": 1,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLeo has the map.\nRyan has the key.\nAlice has the book.\nSwap Leo and Ryan.\nSwap Leo and Alice.\nWho has the map?\nOptions:\nA) Alice\nB) Charlie\nC) Leo\nD) Ryan",
    "answer": "D"
  },
  {
    "id": 2,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 3,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Charlie\nB) David\nC) Lucy\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nRyan has the shell.\nLucy has the lamp.\nSwap Ryan and Lucy.\nWho has the lamp?\nOptions:\nA) Alice\nB) David\nC) Lucy\nD) Ryan",
    "answer": "D"
  },
  {
    "id": 4,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,2)\nWest 1\nSouth 5\nNorth 2\nOptions:\nA) (-1,2)\nB) (0,-1)\nC) (0,2)\nD) (3,2)",
    "answer": "B"
  },
  {
    "id": 5,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 6,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 7,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 8,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT False) OR False\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 9,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()()()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 10,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 11,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 12,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 13,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nEast 2\nEast 1\nSouth 3\nWest 2\nEast 5\nOptions:\nA) (5,-4)\nB) (7,-5)\nC) (8,-3)\nD) (8,-6)",
    "answer": "C"
  },
  {
    "id": 14,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 15,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 16,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nNorth 4\nNorth 5\nNorth 2\nSouth 2\nEast 1\nOptions:\nA) (2,8)\nB) (3,10)\nC) (3,8)\nD) (4,13)",
    "answer": "B"
  },
  {
    "id": 17,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Charlie\nB) Ben\nC) Carl\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nBen has the sword.\nCarl has the ball.\nSwap Ben and Carl.\nWho has the sword?\nOptions:\nA) Alice\nB) Ben\nC) Carl\nD) Peter",
    "answer": "C"
  },
  {
    "id": 18,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nEast 5\nNorth 2\nSouth 1\nEast 2\nOptions:\nA) (11,7)\nB) (5,5)\nC) (8,6)\nD) (9,6)",
    "answer": "C"
  },
  {
    "id": 19,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND NOT True) OR NOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 20,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 21,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "answer": "D"
  },
  {
    "id": 22,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 23,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()(())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 24,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 25,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 4\nEast 4\nNorth 3\nNorth 4\nSouth 3\nOptions:\nA) (10,9)\nB) (7,13)\nC) (9,12)\nD) (9,13)",
    "answer": "C"
  },
  {
    "id": 26,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 2\nWest 4\nEast 4\nOptions:\nA) (4,3)\nB) (6,6)\nC) (7,3)\nD) (9,5)",
    "answer": "B"
  },
  {
    "id": 27,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((())(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 28,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 29,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nNorth 3\nSouth 5\nEast 3\nOptions:\nA) (3,1)\nB) (6,1)\nC) (8,0)\nD) (9,4)",
    "answer": "B"
  },
  {
    "id": 30,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 31,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n))()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 32,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 33,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKai has the badge.\nDavid has the pen.\nAnna has the notebook.\nSwap Kai and Anna.\nSwap Kai and David.\nWho has the pen?\nOptions:\nA) Anna\nB) David\nC) Kai\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 34,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 35,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nNorth 3\nSouth 1\nNorth 2\nEast 3\nOptions:\nA) (4,8)\nB) (5,9)\nC) (6,10)\nD) (6,6)",
    "answer": "B"
  },
  {
    "id": 36,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "answer": "C"
  },
  {
    "id": 37,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Alice\nC) Carl\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the mirror.\nSarah has the notebook.\nSwap Carl and Sarah.\nWho has the notebook?\nOptions:\nA) Bob\nB) Alice\nC) Carl\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 38,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nBob has the pen.\nAmy has the wallet.\nLucy has the flask.\nSwap Amy and Lucy.\nSwap Bob and Amy.\nWho has the wallet?\nOptions:\nA) Amy\nB) Bob\nC) Lucy\nD) Tom",
    "answer": "C"
  },
  {
    "id": 39,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Charlie\nB) Alice\nC) David\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nDavid has the ring.\nMike has the key.\nSwap David and Mike.\nWho has the key?\nOptions:\nA) Bob\nB) Alice\nC) David\nD) Mike",
    "answer": "C"
  },
  {
    "id": 40,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 3\nWest 4\nEast 2\nSouth 2\nWest 5\nOptions:\nA) (-6,-2)\nB) (-6,-5)\nC) (-6,-6)\nD) (-8,-5)",
    "answer": "B"
  },
  {
    "id": 41,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nWest 1\nEast 1\nWest 2\nWest 1\nWest 3\nOptions:\nA) (-3,0)\nB) (-4,3)\nC) (-6,3)\nD) (-6,6)",
    "answer": "B"
  },
  {
    "id": 42,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,5)\nNorth 1\nSouth 4\nNorth 2\nOptions:\nA) (1,2)\nB) (1,6)\nC) (3,4)\nD) (3,7)",
    "answer": "C"
  },
  {
    "id": 43,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,3)\nWest 5\nEast 4\nSouth 5\nOptions:\nA) (0,-4)\nB) (0,1)\nC) (3,-2)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 44,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nKate has the bag.\nOmar has the flask.\nFinn has the wallet.\nSwap Omar and Finn.\nSwap Kate and Finn.\nWho has the wallet?\nOptions:\nA) David\nB) Finn\nC) Kate\nD) Omar",
    "answer": "D"
  },
  {
    "id": 45,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 46,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 47,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 48,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nIvy has the ball.\nPeter has the book.\nSwap Ivy and Peter.\nSwap Sarah and Ivy.\nWho has the key?\nOptions:\nA) Ivy\nB) John\nC) Peter\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 49,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 3\nEast 4\nWest 1\nOptions:\nA) (7,7)\nB) (8,3)\nC) (8,6)\nD) (9,3)",
    "answer": "C"
  },
  {
    "id": 50,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAlice has the bag.\nJack has the cube.\nPeter has the glove.\nSwap Alice and Peter.\nSwap Alice and Jack.\nWho has the bag?\nOptions:\nA) Alice\nB) Finn\nC) Jack\nD) Peter",
    "answer": "D"
  },
  {
    "id": 51,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,0)\nWest 3\nNorth 2\nEast 3\nOptions:\nA) (1,1)\nB) (1,3)\nC) (1,4)\nD) (2,2)",
    "answer": "D"
  },
  {
    "id": 52,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "answer": "D"
  },
  {
    "id": 53,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,4)\nNorth 1\nNorth 2\nWest 2\nOptions:\nA) (0,7)\nB) (0,9)\nC) (1,4)\nD) (2,8)",
    "answer": "A"
  },
  {
    "id": 54,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 55,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 56,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 57,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "answer": "D"
  },
  {
    "id": 58,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 4\nWest 1\nNorth 3\nEast 3\nOptions:\nA) (3,2)\nB) (4,1)\nC) (5,-1)\nD) (8,1)",
    "answer": "C"
  },
  {
    "id": 59,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,5)\nSouth 3\nWest 2\nWest 3\nWest 1\nNorth 5\nOptions:\nA) (-4,5)\nB) (-5,7)\nC) (-6,9)\nD) (-8,8)",
    "answer": "B"
  },
  {
    "id": 60,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 61,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,3)\nSouth 2\nSouth 2\nNorth 3\nOptions:\nA) (3,2)\nB) (4,4)\nC) (5,2)\nD) (5,3)",
    "answer": "A"
  },
  {
    "id": 62,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR True) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 63,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 64,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nNorth 4\nNorth 4\nNorth 3\nOptions:\nA) (4,13)\nB) (5,13)\nC) (6,12)\nD) (7,14)",
    "answer": "A"
  },
  {
    "id": 65,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 66,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nWest 1\nSouth 5\nWest 5\nOptions:\nA) (-1,-5)\nB) (2,-5)\nC) (2,-7)\nD) (2,-8)",
    "answer": "A"
  },
  {
    "id": 67,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSam has the phone.\nLuke has the bag.\nAnna has the badge.\nSwap Sam and Anna.\nSwap Luke and Anna.\nWho has the badge?\nOptions:\nA) Anna\nB) Ivy\nC) Luke\nD) Sam",
    "answer": "D"
  },
  {
    "id": 68,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 69,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((())))())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 70,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the ball.\nSarah has the key.\nAnna has the pen.\nSwap Mike and Anna.\nSwap Sarah and Anna.\nWho has the key?\nOptions:\nA) Anna\nB) Jack\nC) Mike\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 71,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nSouth 4\nSouth 3\nWest 2\nSouth 4\nOptions:\nA) (-2,-7)\nB) (-3,-10)\nC) (0,-9)\nD) (1,-6)",
    "answer": "A"
  },
  {
    "id": 72,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nSouth 5\nNorth 1\nSouth 1\nOptions:\nA) (-1,-2)\nB) (-1,-4)\nC) (-2,-3)\nD) (-4,-4)",
    "answer": "A"
  },
  {
    "id": 73,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND False) OR NOT (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 74,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "answer": "C"
  },
  {
    "id": 75,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nNorth 3\nWest 4\nSouth 3\nWest 2\nEast 4\nOptions:\nA) (-2,0)\nB) (-3,1)\nC) (-3,2)\nD) (-4,0)",
    "answer": "A"
  },
  {
    "id": 76,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 2\nWest 4\nNorth 4\nSouth 4\nOptions:\nA) (1,5)\nB) (2,7)\nC) (4,5)\nD) (4,6)",
    "answer": "D"
  },
  {
    "id": 77,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nSouth 5\nNorth 4\nEast 4\nEast 3\nOptions:\nA) (11,3)\nB) (5,2)\nC) (6,7)\nD) (8,5)",
    "answer": "D"
  },
  {
    "id": 78,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nCharlie has the notebook.\nCarl has the laptop.\nJack has the watch.\nSwap Charlie and Jack.\nSwap Carl and Jack.\nWho has the notebook?\nOptions:\nA) Carl\nB) Charlie\nC) Jack\nD) Lucy",
    "answer": "A"
  },
  {
    "id": 79,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,3)\nSouth 4\nSouth 3\nNorth 2\nOptions:\nA) (3,-5)\nB) (3,0)\nC) (6,-2)\nD) (8,-2)",
    "answer": "C"
  },
  {
    "id": 80,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n((False OR True) AND True) OR False\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 81,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((((()()))))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 82,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nNorth 3\nWest 2\nOptions:\nA) (10,1)\nB) (5,2)\nC) (7,1)\nD) (8,1)",
    "answer": "D"
  },
  {
    "id": 83,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 84,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False AND False) AND (True OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 85,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 86,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 87,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nTom has the phone.\nEmma has the glove.\nEva has the watch.\nSwap Tom and Emma.\nSwap Tom and Eva.\nWho has the watch?\nOptions:\nA) Emma\nB) Eva\nC) Tom\nD) Zara",
    "answer": "C"
  },
  {
    "id": 88,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "answer": "B"
  },
  {
    "id": 89,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nNorth 4\nWest 3\nNorth 5\nEast 3\nOptions:\nA) (2,11)\nB) (3,11)\nC) (4,9)\nD) (6,11)",
    "answer": "B"
  },
  {
    "id": 90,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()())(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 91,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,5)\nWest 5\nSouth 3\nEast 2\nEast 4\nOptions:\nA) (4,5)\nB) (6,2)\nC) (7,5)\nD) (8,2)",
    "answer": "B"
  },
  {
    "id": 92,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 93,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nNorth 4\nNorth 3\nEast 4\nSouth 5\nSouth 1\nOptions:\nA) (2,-1)\nB) (3,0)\nC) (3,1)\nD) (5,2)",
    "answer": "D"
  },
  {
    "id": 94,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,0)\nEast 2\nNorth 2\nEast 3\nOptions:\nA) (11,2)\nB) (12,4)\nC) (13,0)\nD) (9,5)",
    "answer": "A"
  },
  {
    "id": 95,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "answer": "C"
  },
  {
    "id": 96,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 97,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()(())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 98,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 99,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nMike has the flask.\nPeter has the hat.\nCarl has the watch.\nSwap Mike and Carl.\nSwap Peter and Carl.\nWho has the hat?\nOptions:\nA) Carl\nB) Mike\nC) Peter\nD) Sarah",
    "answer": "A"
  },
  {
    "id": 100,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())()(\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 101,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND True) OR NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 102,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,2)\nSouth 5\nSouth 1\nEast 4\nWest 5\nSouth 2\nOptions:\nA) (2,-8)\nB) (3,-4)\nC) (5,-6)\nD) (6,-6)",
    "answer": "C"
  },
  {
    "id": 103,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 4\nWest 4\nSouth 4\nWest 5\nEast 5\nOptions:\nA) (-3,-6)\nB) (0,-3)\nC) (0,-5)\nD) (0,-6)",
    "answer": "D"
  },
  {
    "id": 104,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nEast 4\nSouth 2\nNorth 2\nOptions:\nA) (10,-1)\nB) (8,2)\nC) (8,5)\nD) (9,3)",
    "answer": "B"
  },
  {
    "id": 105,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nSouth 5\nEast 1\nOptions:\nA) (11,-2)\nB) (12,-3)\nC) (8,-1)\nD) (9,-4)",
    "answer": "A"
  },
  {
    "id": 106,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 107,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,2)\nNorth 5\nWest 1\nEast 2\nSouth 1\nOptions:\nA) (2,3)\nB) (2,9)\nC) (3,6)\nD) (4,5)",
    "answer": "C"
  },
  {
    "id": 108,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Charlie\nB) Carl\nC) Mia\nD) Tom",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nTom has the lamp.\nMia has the crown.\nSwap Tom and Mia.\nWho has the lamp?\nOptions:\nA) Alice\nB) Carl\nC) Mia\nD) Tom",
    "answer": "C"
  },
  {
    "id": 109,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nWest 3\nWest 3\nSouth 3\nSouth 5\nOptions:\nA) (-3,-3)\nB) (-5,-3)\nC) (-5,-4)\nD) (-8,-3)",
    "answer": "C"
  },
  {
    "id": 110,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nAmy has the notebook.\nSarah has the crown.\nAlice has the doll.\nSwap Amy and Sarah.\nSwap Amy and Alice.\nWho has the notebook?\nOptions:\nA) Alice\nB) Amy\nC) Kate\nD) Sarah",
    "answer": "D"
  },
  {
    "id": 111,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,6)\nEast 3\nEast 4\nEast 4\nOptions:\nA) (13,3)\nB) (13,5)\nC) (14,9)\nD) (15,6)",
    "answer": "D"
  },
  {
    "id": 112,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nNorth 3\nWest 5\nNorth 1\nOptions:\nA) (-2,10)\nB) (-2,7)\nC) (-5,7)\nD) (-5,8)",
    "answer": "A"
  },
  {
    "id": 113,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nWest 2\nEast 2\nEast 1\nEast 2\nOptions:\nA) (5,5)\nB) (7,3)\nC) (7,7)\nD) (8,8)",
    "answer": "A"
  },
  {
    "id": 114,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nSarah has the key.\nJohn has the ball.\nLuke has the doll.\nSwap Sarah and Luke.\nSwap Sarah and John.\nWho has the key?\nOptions:\nA) David\nB) John\nC) Luke\nD) Sarah",
    "answer": "C"
  },
  {
    "id": 115,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 116,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "answer": "C"
  },
  {
    "id": 117,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()(()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 118,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 119,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Charlie\nB) Carl\nC) Kate\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nCarl has the coin.\nKate has the pen.\nSwap Carl and Kate.\nWho has the pen?\nOptions:\nA) Alice\nB) Carl\nC) Kate\nD) Ryan",
    "answer": "B"
  },
  {
    "id": 120,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nEast 5\nSouth 3\nNorth 1\nOptions:\nA) (11,4)\nB) (13,1)\nC) (8,7)\nD) (9,6)",
    "answer": "A"
  },
  {
    "id": 121,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "answer": "B"
  },
  {
    "id": 122,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()((()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 123,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "answer": "C"
  },
  {
    "id": 124,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 125,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 126,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the sword.\nPeter has the mask.\nJack has the ball.\nSwap Emma and Jack.\nSwap Peter and Jack.\nWho has the mask?\nOptions:\nA) David\nB) Emma\nC) Jack\nD) Peter",
    "answer": "C"
  },
  {
    "id": 127,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((True AND True) AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 128,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nWest 3\nNorth 2\nEast 2\nNorth 3\nNorth 1\nOptions:\nA) (0,9)\nB) (1,8)\nC) (1,9)\nD) (2,8)",
    "answer": "D"
  },
  {
    "id": 129,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nPeter has the cube.\nEva has the hat.\nLucy has the mask.\nSwap Peter and Lucy.\nSwap Peter and Eva.\nWho has the hat?\nOptions:\nA) Eva\nB) Jack\nC) Lucy\nD) Peter",
    "answer": "D"
  },
  {
    "id": 130,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 131,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Charlie\nB) Ivy\nC) Luke\nD) Omar",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nOmar has the hat.\nIvy has the notebook.\nSwap Omar and Ivy.\nWho has the notebook?\nOptions:\nA) Alice\nB) Ivy\nC) Luke\nD) Omar",
    "answer": "D"
  },
  {
    "id": 132,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nSouth 5\nSouth 3\nNorth 1\nWest 1\nOptions:\nA) (1,-6)\nB) (4,-5)\nC) (6,-7)\nD) (7,-8)",
    "answer": "B"
  },
  {
    "id": 133,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,2)\nSouth 1\nEast 5\nEast 5\nSouth 4\nEast 2\nOptions:\nA) (13,-3)\nB) (16,-3)\nC) (16,0)\nD) (17,-4)",
    "answer": "B"
  },
  {
    "id": 134,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT ((False OR False) OR (True AND False))\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 135,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 136,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nJohn has the phone.\nSam has the crown.\nPeter has the wallet.\nSwap John and Sam.\nSwap John and Peter.\nWho has the phone?\nOptions:\nA) Amy\nB) John\nC) Peter\nD) Sam",
    "answer": "D"
  },
  {
    "id": 137,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 138,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 139,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nSouth 2\nEast 4\nSouth 3\nNorth 3\nOptions:\nA) (2,0)\nB) (3,1)\nC) (4,-4)\nD) (5,-1)",
    "answer": "D"
  },
  {
    "id": 140,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 141,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Charlie\nB) Eva\nC) Kate\nD) Mia",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Alice\nB) Eva\nC) Kate\nD) Mia",
    "answer": "C"
  },
  {
    "id": 142,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nNorth 2\nEast 5\nNorth 4\nEast 2\nOptions:\nA) (10,6)\nB) (11,6)\nC) (7,8)\nD) (8,9)",
    "answer": "A"
  },
  {
    "id": 143,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 144,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(50/5)-(3*2)\nOptions:\nA) 0\nB) 2\nC) 3\nD) 4",
    "answer": "D"
  },
  {
    "id": 145,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 146,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "answer": "C"
  },
  {
    "id": 147,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nWest 4\nNorth 2\nSouth 3\nOptions:\nA) (-4,2)\nB) (-4,4)\nC) (-4,5)\nD) (-7,3)",
    "answer": "B"
  },
  {
    "id": 148,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nEast 2\nNorth 4\nWest 2\nEast 4\nOptions:\nA) (10,9)\nB) (5,6)\nC) (6,4)\nD) (7,6)",
    "answer": "D"
  },
  {
    "id": 149,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nEmma has the laptop.\nOmar has the glove.\nLucy has the map.\nSwap Emma and Omar.\nSwap Omar and Lucy.\nWho has the glove?\nOptions:\nA) Emma\nB) Lucy\nC) Omar\nD) Sam",
    "answer": "A"
  },
  {
    "id": 150,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 1\nSouth 1\nNorth 1\nOptions:\nA) (-3,-4)\nB) (-3,1)\nC) (0,-1)\nD) (3,-2)",
    "answer": "C"
  },
  {
    "id": 151,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 2\nSouth 5\nNorth 5\nWest 5\nOptions:\nA) (-4,5)\nB) (-5,6)\nC) (-5,7)\nD) (-6,8)",
    "answer": "B"
  },
  {
    "id": 152,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 3\nWest 2\nSouth 2\nEast 1\nEast 2\nOptions:\nA) (2,1)\nB) (2,2)\nC) (4,2)\nD) (5,3)",
    "answer": "C"
  },
  {
    "id": 153,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 154,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 155,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 156,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nEast 1\nSouth 3\nWest 3\nOptions:\nA) (-1,-2)\nB) (-2,1)\nC) (-2,3)\nD) (-3,1)",
    "answer": "B"
  },
  {
    "id": 157,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT True AND NOT False) OR True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 158,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Charlie\nB) David\nC) Kai\nD) Kate",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nKai has the guitar.\nKate has the camera.\nSwap Kai and Kate.\nWho has the guitar?\nOptions:\nA) Alice\nB) David\nC) Kai\nD) Kate",
    "answer": "D"
  },
  {
    "id": 159,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())))()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 160,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,6)\nSouth 2\nNorth 2\nEast 5\nWest 2\nOptions:\nA) (3,9)\nB) (5,5)\nC) (6,6)\nD) (9,8)",
    "answer": "C"
  },
  {
    "id": 161,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,3)\nSouth 3\nWest 5\nWest 2\nWest 3\nSouth 3\nOptions:\nA) (-11,-1)\nB) (-8,-3)\nC) (-9,-1)\nD) (-9,0)",
    "answer": "B"
  },
  {
    "id": 162,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nSouth 2\nSouth 4\nSouth 5\nSouth 1\nOptions:\nA) (0,-15)\nB) (1,-12)\nC) (3,-12)\nD) (5,-9)",
    "answer": "C"
  },
  {
    "id": 163,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,5)\nSouth 1\nEast 4\nNorth 4\nWest 3\nOptions:\nA) (-1,11)\nB) (1,8)\nC) (3,6)\nD) (4,5)",
    "answer": "B"
  },
  {
    "id": 164,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "answer": "C"
  },
  {
    "id": 165,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n)()()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 166,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,1)\nEast 5\nSouth 5\nSouth 5\nOptions:\nA) (5,-8)\nB) (6,-8)\nC) (7,-9)\nD) (8,-12)",
    "answer": "C"
  },
  {
    "id": 167,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,6)\nSouth 5\nSouth 3\nNorth 3\nWest 2\nEast 2\nOptions:\nA) (4,1)\nB) (6,1)\nC) (8,-2)\nD) (8,2)",
    "answer": "B"
  },
  {
    "id": 168,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n18/(3+3)\nOptions:\nA) -1\nB) 1\nC) 2\nD) 3",
    "answer": "D"
  },
  {
    "id": 169,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "answer": "C"
  },
  {
    "id": 170,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 171,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "answer": "C"
  },
  {
    "id": 172,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 173,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 174,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 175,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,0)\nSouth 2\nNorth 1\nWest 3\nSouth 2\nOptions:\nA) (-1,-4)\nB) (-3,-3)\nC) (-5,-1)\nD) (0,0)",
    "answer": "B"
  },
  {
    "id": 176,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 177,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nLucy has the doll.\nCarl has the flask.\nDavid has the pen.\nSwap Lucy and David.\nSwap Lucy and Carl.\nWho has the flask?\nOptions:\nA) Carl\nB) David\nC) Lucy\nD) Sam",
    "answer": "C"
  },
  {
    "id": 178,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "answer": "B"
  },
  {
    "id": 179,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nSouth 3\nNorth 1\nSouth 2\nSouth 5\nNorth 1\nOptions:\nA) (2,-8)\nB) (4,-7)\nC) (6,-7)\nD) (7,-6)",
    "answer": "B"
  },
  {
    "id": 180,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "answer": "C"
  },
  {
    "id": 181,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,4)\nSouth 4\nNorth 4\nSouth 2\nNorth 4\nOptions:\nA) (-1,5)\nB) (0,3)\nC) (0,8)\nD) (1,6)",
    "answer": "D"
  },
  {
    "id": 182,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n())(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "B"
  },
  {
    "id": 183,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,5)\nEast 2\nEast 2\nNorth 2\nEast 4\nOptions:\nA) (10,7)\nB) (11,4)\nC) (13,8)\nD) (7,10)",
    "answer": "A"
  },
  {
    "id": 184,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,6)\nEast 4\nSouth 1\nNorth 1\nWest 1\nEast 2\nOptions:\nA) (4,8)\nB) (6,4)\nC) (6,8)\nD) (7,6)",
    "answer": "D"
  },
  {
    "id": 185,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(NOT False) AND (NOT True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 186,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND True) OR NOT (True AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 187,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 188,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,0)\nSouth 5\nSouth 2\nNorth 2\nOptions:\nA) (0,-6)\nB) (1,-5)\nC) (1,-6)\nD) (4,-6)",
    "answer": "B"
  },
  {
    "id": 189,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR False) AND (True OR True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 190,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the map.\nBob has the shell.\nCarl has the ring.\nSwap Bob and Carl.\nSwap David and Bob.\nWho has the shell?\nOptions:\nA) Bob\nB) Carl\nC) David\nD) Ryan",
    "answer": "B"
  },
  {
    "id": 191,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Charlie\nB) Eva\nC) Finn\nD) Leo",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the sword.\nLeo has the bag.\nSwap Eva and Leo.\nWho has the sword?\nOptions:\nA) Alice\nB) Eva\nC) Finn\nD) Leo",
    "answer": "D"
  },
  {
    "id": 192,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nEast 4\nEast 5\nNorth 2\nWest 5\nWest 3\nOptions:\nA) (2,3)\nB) (2,4)\nC) (3,5)\nD) (4,3)",
    "answer": "D"
  },
  {
    "id": 193,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND NOT (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 194,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 195,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "B"
  },
  {
    "id": 196,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((7-2)*(8-3))\nOptions:\nA) 21\nB) 23\nC) 24\nD) 25",
    "answer": "D"
  },
  {
    "id": 197,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,3)\nWest 2\nEast 4\nNorth 2\nSouth 5\nOptions:\nA) (0,-3)\nB) (1,-1)\nC) (3,-3)\nD) (3,0)",
    "answer": "D"
  },
  {
    "id": 198,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR True) AND (False AND True)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 199,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 200,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,4)\nSouth 3\nSouth 1\nEast 5\nSouth 2\nOptions:\nA) (10,-2)\nB) (11,-1)\nC) (12,1)\nD) (9,-2)",
    "answer": "D"
  }
]"""

tasks = json.loads(raw_tasks_json)
print(f"Loaded {len(tasks)} tasks.")


Loaded 200 tasks.


## Step 3 — Evaluation Logic

Because the answers vary per task, we won't extract just two token probabilities. Instead, we will:
1. Format the prompt using the chat template.
2. Pass it through the model.
3. Check the highest probability token predicted at the final position.
4. Decode it and compare it to the true answer.

## Step 3 — Evaluation Logic

Because the answers vary per task, we won't extract just two token probabilities. Instead, we will:
1. Format the prompt using the chat template.
2. Pass it through the model.
3. Check the highest probability token predicted at the final position.
4. Decode it and compare it to the true answer.

In [40]:
def format_prompt(text):
    # Remove "Answer:" from the user prompt text so it isn't trapped
    text = text.strip()
    if text.endswith("Answer:"):
        text = text[:-7].strip()
        
    messages = [{"role": "user", "content": text}]
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Force the assistant to start its turn with "Answer:"
    prompt_str += "Answer:"
    return prompt_str

def evaluate_prompt(prompt_text, true_answer):
    """
    Returns (predicted_text, is_correct, token_probability)
    """
    formatted_prompt = format_prompt(prompt_text)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = torch.nn.functional.softmax(logits, dim=-1)
        
    # Get the single most likely next token (unconstrained)
    top_token_id = torch.argmax(probs, dim=-1).item()
    top_token_prob = probs[0, top_token_id].item()
    predicted_text = tokenizer.decode([top_token_id]).strip()
    
    # Compare with true answer
    is_correct = predicted_text.upper() == str(true_answer).strip().upper()
    
    return predicted_text, is_correct, top_token_prob


## Step 4 — Run Feasibility Check

In [41]:
results = []

print("=========================================================================")
print(f"{'ID':<3} | {'Task Type':<25} | {'True Ans':<10} | {'0-Shot (Student)':<20} | {'4-Shot (Teacher)':<20}")
print("=========================================================================")

for t in tasks:
    task_id = t["id"]
    task_type = t["task"]
    true_ans = str(t["answer"])
    
    # Evaluate 0-shot Student
    s_pred, s_correct, s_prob = evaluate_prompt(t["student_prompt"], true_ans)
    s_mark = "✅" if s_correct else "❌"
    s_str = f"{s_pred} ({s_prob:.2f}) {s_mark}"
    
    # Evaluate 4-shot Teacher
    t_pred, t_correct, t_prob = evaluate_prompt(t["teacher_prompt"], true_ans)
    t_mark = "✅" if t_correct else "❌"
    t_str = f"{t_pred} ({t_prob:.2f}) {t_mark}"
    
    results.append({
        "id": task_id,
        "task": task_type,
        "teacher_correct": t_correct,
        "student_correct": s_correct,
        "teacher_pred": t_pred,
        "student_pred": s_pred,
        "teacher_prompt_full": format_prompt(t["teacher_prompt"]),
        "student_prompt_full": format_prompt(t["student_prompt"]),
    })
    
    print(f"{task_id:<3} | {task_type:<25} | {true_ans:<10} | {s_str:<20} | {t_str:<20}")

print("=========================================================================")

# Print out the exact prompts and responses for debugging
for r in results:
    print(f"\n{'='*80}")
    print(f"TASK ID: {r['id']} | TYPE: {r['task']}")
    print(f"{'-'*80}")
    print(f"TEACHER (4-Shot) PROMPT:\n{r['teacher_prompt_full']}")
    print(f"\nTEACHER PREDICTED: {r['teacher_pred']} (Correct: {r['teacher_correct']})")
    print(f"{'-'*80}")
    print(f"STUDENT (0-Shot) PROMPT:\n{r['student_prompt_full']}")
    print(f"\nSTUDENT PREDICTED: {r['student_pred']} (Correct: {r['student_correct']})")
    print(f"{'='*80}\n")


ID  | Task Type                 | True Ans   | 0-Shot (Student)     | 4-Shot (Teacher)    
1   | tracking_shuffled_objects | D          | C (0.59) ❌           | C (0.71) ❌          
2   | dyck_language             | A          | B (0.57) ❌           | A (0.60) ✅          
3   | tracking_shuffled_objects | D          | C (0.86) ❌           | C (0.68) ❌          
4   | navigate                  | B          | A (0.46) ❌           | C (0.46) ❌          
5   | dyck_language             | A          | B (0.57) ❌           | A (0.63) ✅          
6   | boolean_expressions       | A          | B (0.58) ❌           | A (0.66) ✅          
7   | boolean_expressions       | B          | B (0.82) ✅           | B (0.58) ✅          
8   | boolean_expressions       | B          | B (0.78) ✅           | A (0.52) ❌          
9   | dyck_language             | B          | B (0.61) ✅           | B (0.63) ✅          
10  | dyck_language             | B          | B (0.66) ✅           | B (0.65) ✅          

In [42]:
golden_tasks = [r for r in results if r["teacher_correct"] and not r["student_correct"]]
teacher_failures = [r for r in results if not r["teacher_correct"]]
student_successes = [r for r in results if r["student_correct"]]

print(f"Total Tasks Evaluated : {len(tasks)}")
print(f"Teacher Failures      : {len(teacher_failures)} (Teacher couldn't solve with 4 shots)")
print(f"Student Successes     : {len(student_successes)} (Student solved 0-shot; no gap to close)")
print(f"GOLDEN TASKS          : {len(golden_tasks)} (Perfect for distillation)")

if len(golden_tasks) > 0:
    print("\nGolden Task IDs:", [r["id"] for r in golden_tasks])
else:
    print("\n⚠️ No golden tasks found. We need to rethink the prompts or model size.")

Total Tasks Evaluated : 200
Teacher Failures      : 110 (Teacher couldn't solve with 4 shots)
Student Successes     : 76 (Student solved 0-shot; no gap to close)
GOLDEN TASKS          : 45 (Perfect for distillation)

Golden Task IDs: [2, 5, 6, 14, 15, 21, 24, 30, 34, 36, 46, 52, 55, 63, 65, 74, 86, 88, 92, 95, 98, 106, 116, 121, 123, 124, 125, 130, 135, 138, 140, 145, 146, 153, 155, 164, 169, 171, 172, 173, 174, 178, 180, 194, 199]


In [44]:
# ── Golden Task Breakdown ──
# Look up true answers from the tasks list (results dict has no 'answer' key)
answer_lookup = {t['id']: t['answer'] for t in tasks}

print(f'\n{"="*75}')
print(f'{"ID":<6} {"Type":<30} {"Correct":>8} {"Student":>9} {"Teacher":>9}')
print(f'{"="*75}')
for r in golden_tasks:
    true_ans = answer_lookup.get(r['id'], '?')
    print(f"{r['id']:<6} {r['task']:<30} {true_ans:>8} "
          f"{r['student_pred']:>4} ❌   {r['teacher_pred']:>4} ✅")
print(f'{"="*75}')
print(f'\nTotal golden tasks: {len(golden_tasks)}')



ID     Type                            Correct   Student   Teacher
2      dyck_language                         A    B ❌      A ✅
5      dyck_language                         A    B ❌      A ✅
6      boolean_expressions                   A    B ❌      A ✅
14     dyck_language                         A    B ❌      A ✅
15     boolean_expressions                   A    B ❌      A ✅
21     multistep_arithmetic                  D    B ❌      D ✅
24     dyck_language                         A    B ❌      A ✅
30     navigate                              C    B ❌      C ✅
34     boolean_expressions                   A    B ❌      A ✅
36     navigate                              C    B ❌      C ✅
46     dyck_language                         A    B ❌      A ✅
52     multistep_arithmetic                  D    B ❌      D ✅
55     boolean_expressions                   A    B ❌      A ✅
63     dyck_language                         A    B ❌      A ✅
65     boolean_expressions                   A    

In [ ]:
              [8, 15, 16, 20, 22, 27, 30, 31, 35, 41, 50]

[8, 15, 16, 20, 22, 27, 30, 31, 35, 41, 50]